# ROSACE Python – OCT1 DMS Real Data

This notebook ports the R `intro_rosace` vignette to Python using the **actual
OCT1 deep mutational scanning (DMS) data** stored in `data/R/oct1.rda`.  It
mirrors the workflow from the [R intro_rosace
vignette](https://github.com/pimentellab/rosace/blob/main/vignettes/intro_rosace.Rmd).

### Pipeline overview
1. Load `oct1.rda` with `rdata` (file copied to a temp location first)
2. Convert each replicate DataFrame to an `AssayGrowth` object
3. Filter, impute, and normalize each replicate
4. Integrate all three replicates into an `AssaySetGrowth`
5. Score variants with Simple Linear Regression (SLR)
6. Hypothesis-test and label variants (Neg / Neutral / Pos)
7. Explore top loss-of-function and gain-of-function variants
8. Visualize score distributions


In [ ]:
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

import re
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rdata

from rosace.assay import AssayGrowth, AssaySetGrowth
from rosace.preprocessing import filter_data, impute_data, normalize_data
from rosace.slr import run_slr
from rosace.utils import output_score
from rosace.visualization import score_density


## 1. Load `oct1.rda`

The `.rda` file is copied to a temporary directory before being read, so the
original data files are never modified.  The three resulting DataFrames
(`oct1_rep1`, `oct1_rep2`, `oct1_rep3`) each contain an `hgvs` column of
variant names and count columns `c_0`, `c_2`, `c_4` (and `c_6` for replicates
1 and 2).  A single row named `_wt` holds the summed wildtype counts.


In [ ]:
# Locate the .rda file relative to this notebook
notebook_dir = Path(".").resolve()
repo_root = notebook_dir.parent
rda_src = repo_root / "data" / "R" / "oct1.rda"

# Copy to a temporary directory so we work from a clean location
tmp_dir = Path(tempfile.mkdtemp())
rda_path = shutil.copy(str(rda_src), str(tmp_dir))
print(f"Working copy: {rda_path}")

# Read all objects from the .rda file
oct1_data = rdata.read_rda(rda_path)
print(f"Objects in .rda: {list(oct1_data.keys())}")


In [ ]:
# Preview each replicate DataFrame
for key, df in oct1_data.items():
    df = df.copy()
    df.columns = [str(c) for c in df.columns]
    count_cols = [c for c in df.columns if c.startswith("c_")]
    print(f"{key}: {df.shape[0]} variants, timepoints = {count_cols}")
    display(df.head(3))
    print()


## 2. Build `AssayGrowth` Objects

Each replicate is wrapped in an `AssayGrowth`.  Note that replicate 3 has only
**3 timepoints** (`c_0`, `c_2`, `c_4`) while replicates 1 and 2 have **4**
(`c_0`, `c_2`, `c_4`, `c_6`).  The framework handles mixed timepoint counts
automatically via an outer join during integration.


In [ ]:
def make_assay(df_raw: pd.DataFrame, rep: int) -> AssayGrowth:
    """Convert a replicate DataFrame from oct1.rda into an AssayGrowth."""
    df = df_raw.copy()
    df.columns = [str(c) for c in df.columns]
    count_cols = [c for c in df.columns if c.startswith("c_")]
    return AssayGrowth(
        counts=df[count_cols].values.astype(float),
        var_names=df["hgvs"].tolist(),
        key="OCT1",
        rep=rep,
    )


raw_assays = [
    make_assay(oct1_data["oct1_rep1"], rep=1),
    make_assay(oct1_data["oct1_rep2"], rep=2),
    make_assay(oct1_data["oct1_rep3"], rep=3),
]

for a in raw_assays:
    print(f"Replicate {a.rep}: {a.counts.shape[0]} variants × {a.counts.shape[1]} timepoints")


## 3. Filter, Impute, and Normalize

### 3.1 Filtering

Remove variants with:
- More than 50 % missing timepoints (`na_rmax=0.5`)
- Total count < 20 across all timepoints (`min_count=20`)


In [ ]:
filtered_assays = []
for a in raw_assays:
    fa = filter_data(a, na_rmax=0.5, min_count=20)
    print(f"Rep {a.rep}: {a.counts.shape[0]} → {fa.counts.shape[0]} variants after filtering")
    filtered_assays.append(fa)


### 3.2 Imputation

Replace remaining `NaN` values with zero (zero-imputation).


In [ ]:
imputed_assays = []
for a in filtered_assays:
    ia = impute_data(a, method="zero")
    nan_count = np.isnan(ia.counts).sum()
    print(f"Rep {a.rep}: {nan_count} NaN values remaining after imputation")
    imputed_assays.append(ia)


### 3.3 Normalization

Wildtype normalization uses the single `_wt` control row:

```
norm = log(count + 0.5) − log(wt_count + 0.5) − baseline_at_t0
```

The `_wt` row is removed from the output (`wt_rm=True`).


In [ ]:
norm_assays = []
for a in imputed_assays:
    na = normalize_data(a, method="wt", wt_var_names=["_wt"], wt_rm=True)
    print(f"Rep {a.rep}: {na.norm_counts.shape[0]} variants in norm_counts (wt removed)")
    norm_assays.append(na)


## 4. Integrate Three Replicates into `AssaySetGrowth`

All three normalized replicates are combined via a full outer join.  Variants
present in only some replicates receive `NaN` for the missing replicate's
columns.


In [ ]:
def integrate_replicates(assays: list[AssayGrowth]) -> AssaySetGrowth:
    """Integrate any number of normalized AssayGrowth objects via outer join."""
    def norm_df(a: AssayGrowth) -> pd.DataFrame:
        names = a.norm_var_names or a.var_names
        T = a.norm_counts.shape[1]
        return pd.DataFrame(
            a.norm_counts,
            index=names,
            columns=[f"r{a.rep}_t{t}" for t in range(T)],
        )

    def raw_df(a: AssayGrowth) -> pd.DataFrame:
        T = a.counts.shape[1]
        return pd.DataFrame(
            a.counts,
            index=a.var_names,
            columns=[f"r{a.rep}_raw_t{t}" for t in range(T)],
        )

    combined = norm_df(assays[0])
    for a in assays[1:]:
        combined = combined.join(norm_df(a), how="outer")

    raw_combined = raw_df(assays[0])
    for a in assays[1:]:
        raw_combined = raw_combined.join(raw_df(a), how="outer")

    return AssaySetGrowth(
        combined_counts=combined.values,
        var_names=list(combined.index),
        reps=[a.rep for a in assays],
        key=assays[0].key,
        raw_counts=raw_combined.values,
        rounds=[a.rounds for a in assays],
    )


assay_set = integrate_replicates(norm_assays)
print(
    f"AssaySetGrowth: {assay_set.combined_counts.shape[0]} variants, "
    f"{assay_set.combined_counts.shape[1]} columns "
    f"(reps {assay_set.reps}, rounds per rep: {assay_set.rounds})"
)


## 5. Parse Variant Metadata

HGVS-style names like `p.(A107C)` encode the wildtype residue, position, and
mutant residue.  Synonymous variants (e.g. `p.(A107A)`) and deletion variants
(e.g. `p.(A107del)`) are also present in the data.


In [ ]:
_HGVS_RE = re.compile(r"p\.\(([A-Z])(\d+)([A-Z])\)")


def parse_hgvs(hgvs: str) -> tuple:
    """Return (wt_aa, position, mut_aa, variant_type) from an HGVS name."""
    m = _HGVS_RE.match(hgvs)
    if m:
        wt_aa, pos, mut_aa = m.group(1), int(m.group(2)), m.group(3)
        vtype = "synonymous" if wt_aa == mut_aa else "missense"
        return wt_aa, pos, mut_aa, vtype
    if "del" in hgvs:
        return None, None, None, "deletion"
    return None, None, None, "other"


parsed = [parse_hgvs(v) for v in assay_set.var_names]
var_meta = pd.DataFrame(
    parsed,
    columns=["wt", "position", "mut", "type"],
    index=assay_set.var_names,
).rename_axis("variant").reset_index()

print("Variant type distribution:")
print(var_meta["type"].value_counts())
var_meta.head()


## 6. Score Variants with Simple Linear Regression (SLR)

SLR fits an OLS regression of normalized count vs. timepoint index for each
variant.  The **slope** is used as the functional score and serves as a fast
baseline before the full Bayesian ROSACE model.


In [ ]:
score_obj = run_slr(assay_set)
print(f"Method : {score_obj.method}")
print(f"Type   : {score_obj.type}")
print(f"Assay  : {score_obj.assay_name}")
print(f"Scored variants: {len(score_obj.score)}")
score_obj.score.head()


## 7. Hypothesis Testing

`output_score` computes the **local false sign rate (LFSR)** and labels
variants at `sig_test = 0.05`:

| LFSR       | Label   |
|------------|---------|
| ≥ 0.05     | Neutral |
| < 0.05, mean > 0 | Pos |
| < 0.05, mean ≤ 0 | Neg |


In [ ]:
scores = output_score(score_obj, sig_test=0.05)
print("Label distribution:")
print(scores["label"].value_counts())
scores.head()


## 8. Merge Scores with Variant Metadata


In [ ]:
scores_meta = scores.merge(var_meta, on="variant", how="left")
print(f"Merged DataFrame shape: {scores_meta.shape}")
scores_meta.head()


## 9. Top Loss-of-Function and Gain-of-Function Variants


In [ ]:
missense = scores_meta[scores_meta["type"] == "missense"].copy()
cols = ["variant", "position", "wt", "mut", "mean", "sd", "lfsr", "label"]

print("Top 10 Loss-of-Function variants (most negative SLR score):")
display(missense.nsmallest(10, "mean")[cols].reset_index(drop=True))

print("\nTop 10 Gain-of-Function variants (most positive SLR score):")
display(missense.nlargest(10, "mean")[cols].reset_index(drop=True))


## 10. Position-Level Summary

Aggregate per-position mean score and the fraction of negatively / positively
selected variants.


In [ ]:
pos_summary = (
    missense.dropna(subset=["position"])
    .groupby("position")
    .agg(
        mean_score=("mean", "mean"),
        n_variants=("variant", "count"),
        n_neg=("label", lambda s: (s == "Neg").sum()),
        n_pos=("label", lambda s: (s == "Pos").sum()),
    )
    .reset_index()
)
pos_summary["frac_neg"] = pos_summary["n_neg"] / pos_summary["n_variants"]
pos_summary["frac_pos"] = pos_summary["n_pos"] / pos_summary["n_variants"]

print("Top 10 positions by most negative mean score:")
display(pos_summary.nsmallest(10, "mean_score").reset_index(drop=True))


## 11. Score Distribution Visualization

`score_density` plots KDE density curves of SLR functional scores, separated
by variant type (missense vs. synonymous).


In [ ]:
plot_data = scores_meta[scores_meta["type"].isin(["missense", "synonymous"])].copy()

fig = score_density(
    data=plot_data,
    type_col="type",
    score_col="mean",
    order=["missense", "synonymous"],
    title="SLR Score Distribution – OCT1 DMS (real data)",
    show=False,
)
plt.tight_layout()
plt.show()


## 12. Summary Statistics


In [ ]:
print("=== OCT1 DMS Analysis Summary ===")
print(f"  Total variants scored:  {len(scores_meta):>6}")
print(f"  Missense:               {(scores_meta['type'] == 'missense').sum():>6}")
print(f"  Synonymous:             {(scores_meta['type'] == 'synonymous').sum():>6}")
print(f"  Deletions:              {(scores_meta['type'] == 'deletion').sum():>6}")
print()
print("SLR label distribution (all variants):")
print(scores_meta["label"].value_counts().to_string())
print()
print("SLR label distribution (missense only):")
print(scores_meta.loc[scores_meta["type"] == "missense", "label"].value_counts().to_string())


## Summary

This notebook demonstrated the complete ROSACE Python workflow on the **real
OCT1 DMS dataset** loaded directly from the R `.rda` file:

1. **Data loading** – `oct1.rda` read with `rdata`, three replicate DataFrames
   extracted and converted to `AssayGrowth` objects
2. **QC** – per-replicate filtering (missingness + minimum counts), zero
   imputation, wildtype normalization
3. **Integration** – three replicates (with different timepoint counts) merged
   into a single `AssaySetGrowth` via outer join
4. **SLR scoring** – fast linear-regression baseline functional scores
5. **Hypothesis testing** – LFSR-based Neg / Neutral / Pos labels
6. **Interpretation** – top LOF/GOF variants, position-level summaries
7. **Visualisation** – score density plots by mutation type

For the full Bayesian ROSACE analysis, replace `run_slr(assay_set)` with
`run_rosace(assay_set, method="ROSACE1", var_info=var_info)` (requires
CmdStan and `cmdstanpy`).
